# LangChain Agent Under the Hood

Welcome to this lesson on understanding what happens inside a LangChain agent.

When you call:

```python
agent.invoke(...)
```

it may look like a single line of code.

However, behind this simple call, a complete reasoning and tool-execution loop is happening.

In this lesson, we will peel back the layers and understand the communication between:

```text
User
   ↓
Your Python Application
   ↓
LangChain
   ↓
LLM / Model API
   ↓
LangChain
   ↓
Python Tool
   ↓
LangChain
   ↓
LLM / Model API
   ↓
Final Answer
```

We will use a simple mathematical question:

> What is 15 multiplied by 8 and then divided by 3?

The question is intentionally simple because it allows us to clearly understand the underlying architecture.

The same architecture can be used for much more complex operations such as database queries, API calls, file operations, emails, and cloud infrastructure.

---

# 1. The Simple Agent Code

Previously, we created an agent using something similar to:

```python
agent = create_agent(
    model=model,
    tools=tools,
)
```

Then we invoked it:

```python
result = agent.invoke(...)
```

From the developer's perspective, this looks very simple.

But internally, the agent performs many operations.

The basic flow is:

```text
agent.invoke()
       ↓
Send request to LLM
       ↓
LLM decides what to do
       ↓
Tool call requested
       ↓
LangChain executes tool
       ↓
Tool result returned
       ↓
Send result back to LLM
       ↓
LLM decides whether more work is needed
       ↓
Repeat if necessary
       ↓
Final answer
```

This is the ReAct pattern:

```text
Reason
   ↓
Act
   ↓
Observe
   ↓
Repeat
```

---

# 2. Step 1: The User Runs Python Code

Everything starts with the user.

The user asks:

> What is 15 multiplied by 8 and then divided by 3?

Your Python application receives this question.

Your code then calls:

```python
agent.invoke(...)
```

This is the entry point into the LangChain agent loop.

The control flow is:

```text
User
 ↓
Your Python Code
 ↓
LangChain
 ↓
Model API
```

An important point is that the user does **not** communicate directly with the LLM.

Instead:

```text
User
 ↓
Application
 ↓
LangChain
 ↓
LLM
```

Your application remains in control.

It decides:

* Which model to use
* Which tools are available
* What data the tools can access
* How tool results are handled
* What safety rules are applied
* What the final response looks like

This separation is extremely important in production AI systems.

---

# 3. Step 2: LangChain Calls the Model

When LangChain sends the request to the model API, it typically provides two important things.

## 1. Messages

The messages represent the conversation so far.

Initially, this might simply be:

```json
{
  "role": "user",
  "content": "What is 15 multiplied by 8 and then divided by 3?"
}
```

## 2. Tools

LangChain also provides the model with information about the tools it is allowed to use.

For example:

```json
{
  "name": "multiply",
  "description": "Multiply two numbers together.",
  "parameters": {
    "type": "object",
    "properties": {
      "a": {
        "type": "number"
      },
      "b": {
        "type": "number"
      }
    },
    "required": ["a", "b"]
  }
}
```

The model now knows:

```text
Available Tool
     ↓
multiply

Description
     ↓
Multiply two numbers

Inputs
     ↓
a: number
b: number
```

The model can use this information to decide whether the tool is appropriate.

---

# 4. Where Does the Tool Schema Come From?

You may remember defining a Python function like this:

```python
@tool
def multiply(a: float, b: float) -> float:
    """
    Multiply two numbers together.
    """
    return a * b
```

You wrote Python code.

However, the LLM does not directly receive your Python function.

Instead, LangChain examines:

* Function name
* Docstring
* Type hints
* Function parameters
* Return type

It then converts this information into a structured tool schema.

Conceptually:

```text
Python Function
      ↓
LangChain
      ↓
Tool Schema
      ↓
LLM
```

This is why tool definitions are so important.

A clear function name and description help the model make better tool-selection decisions.

---

# 5. Step 3: The Model Reasons About the Question

Now the model receives:

```text
Question:
What is 15 × 8 ÷ 3?

Available Tools:
- multiply
- divide
- add
```

The model determines that the operation should happen in stages.

First:

```text
15 × 8
```

Then:

```text
Result ÷ 3
```

The model decides that it should call the `multiply` tool first.

However, the model cannot directly execute your Python function.

The model does not run:

```python
multiply(15, 8)
```

Instead, it returns a structured tool-call request.

Conceptually:

```json
{
  "tool_calls": [
    {
      "id": "call_001",
      "name": "multiply",
      "arguments": {
        "a": 15,
        "b": 8
      }
    }
  ]
}
```

The important information here is:

```text
Tool Name
    ↓
multiply

Arguments
    ↓
a = 15
b = 8

Call ID
    ↓
call_001
```

The `call_id` is important because it allows the framework to associate the tool result with the correct tool request.

At this point, the model has **not** provided the final answer.

It has requested an action.

---

# 6. Step 4: LangChain Interprets the Tool Call

LangChain receives the model's response.

It checks:

```text
Does the model have a final text answer?
        ↓
       No

Does the model have a tool call?
        ↓
       Yes
```

LangChain then extracts:

```text
Tool Name
     ↓
multiply

Arguments
     ↓
a = 15
b = 8

Call ID
     ↓
call_001
```

The framework handles this parsing automatically.

Without a framework, you would have to write this logic yourself.

You would need to:

1. Parse the model response.
2. Detect whether a tool call exists.
3. Extract the tool name.
4. Parse the arguments.
5. Find the correct Python function.
6. Execute it.
7. Format the result.
8. Send the result back to the model.

Frameworks such as LangChain automate much of this process.

---

# 7. Step 5: Mapping the Tool Name to Python

Now we have an interesting problem.

The model says:

```text
Call "multiply"
```

But `"multiply"` is just a string.

How does LangChain know which Python function to execute?

LangChain maintains an internal mapping between tool names and actual Python functions.

Conceptually:

```python
tool_registry = {
    "multiply": multiply,
    "divide": divide,
    "add": add,
}
```

So when the model says:

```text
multiply
```

LangChain performs a lookup:

```python
tool_registry["multiply"]
```

The result is the actual Python function:

```python
multiply
```

This creates a bridge between two worlds.

### Model World

```text
Text
JSON
Tool Names
Arguments
```

### Python World

```text
Functions
Objects
Code Execution
Return Values
```

The framework connects these two worlds.

```text
LLM
 ↓
"multiply"
 ↓
Tool Registry
 ↓
Python multiply() Function
```

This is a critical part of agent architecture.

---

# 8. Step 6: Python Code Executes

Now LangChain has identified the correct Python function.

The actual function runs:

```python
def multiply(a: float, b: float) -> float:
    return a * b
```

LangChain calls:

```python
multiply(15, 8)
```

Python executes the calculation:

```text
15 × 8 = 120
```

The result is:

```text
120
```

Notice something important.

At this stage, **no AI reasoning is happening**.

This is normal Python code execution.

The model decided:

> "I need multiplication."

LangChain executed:

```python
multiply(15, 8)
```

Python returned:

```text
120
```

The separation is:

```text
LLM
Decision
   ↓
LangChain
Orchestration
   ↓
Python
Execution
```

---

# 9. Why This Matters in Real Applications

Our example is very simple.

We are multiplying two numbers.

However, the exact same architecture can be used for powerful operations.

For example:

```text
LLM
 ↓
"Query Customer Database"
 ↓
LangChain
 ↓
Database Tool
 ↓
SQL Query
 ↓
Database Result
 ↓
LLM
```

Or:

```text
LLM
 ↓
"Check Weather"
 ↓
Weather API
 ↓
Current Weather
 ↓
LLM
```

Or:

```text
LLM
 ↓
"Create Support Ticket"
 ↓
Ticketing API
 ↓
Ticket Created
 ↓
LLM
```

Or:

```text
LLM
 ↓
"Send Email"
 ↓
Email API
 ↓
Email Sent
 ↓
LLM
```

The pattern remains the same.

The only thing that changes is the tool.

The simple math function:

```python
multiply(15, 8)
```

could be replaced by:

```python
query_database(...)
```

or:

```python
call_external_api(...)
```

or:

```python
send_email(...)
```

The architecture is identical.

---

# 10. Step 7: Send the Tool Result Back

After the Python function executes, LangChain needs to send the result back to the model.

The model needs context.

The conversation now contains multiple messages.

Conceptually:

```text
1. User Message
   ↓
"What is 15 × 8 ÷ 3?"

2. Assistant Tool Call
   ↓
"Call multiply(15, 8)"

3. Tool Result
   ↓
"120"
```

The tool result is associated with the original tool-call ID:

```text
call_001
```

Conceptually:

```json
{
  "role": "tool",
  "tool_call_id": "call_001",
  "content": "120"
}
```

The `tool_call_id` connects the result with the original request.

This tells the model:

> You requested the `multiply` operation with call ID `call_001`. The result is `120`.

---

# 11. Why Is the Conversation Sent Again?

LLMs are generally stateless between API requests.

This means the model does not automatically remember the previous request.

Therefore, when LangChain sends the next request, it provides the relevant conversation history again.

Conceptually:

```text
User:
15 × 8 ÷ 3

Assistant:
Call multiply(15, 8)

Tool:
120
```

The model now has enough context to continue reasoning.

It can understand:

```text
I asked for multiplication.
The application executed it.
The result is 120.
The original user also wants division by 3.
```

The model can now continue.

---

# 12. Step 8: The Model Requests the Second Tool

The model receives the updated conversation.

It now sees:

```text
15 × 8 ÷ 3

Multiply Result:
120
```

The model reasons that the next operation is:

```text
120 ÷ 3
```

So it generates another tool call.

Conceptually:

```json
{
  "tool_calls": [
    {
      "id": "call_002",
      "name": "divide",
      "arguments": {
        "a": 120,
        "b": 3
      }
    }
  ]
}
```

The process repeats.

```text
LLM
 ↓
Request divide(120, 3)
 ↓
LangChain
 ↓
Find divide Python function
 ↓
Execute divide(120, 3)
 ↓
Result = 40
 ↓
Send result back to LLM
```

The model then receives:

```text
40
```

At this point, the model has enough information to provide the final answer.

---

# 13. The Complete Internal Flow

The entire process looks like this:

```text
USER
"What is 15 × 8 ÷ 3?"
        │
        ▼
YOUR PYTHON CODE
agent.invoke()
        │
        ▼
LANGCHAIN
Build messages + tool schemas
        │
        ▼
LLM
Decides: Call multiply
        │
        ▼
TOOL CALL
multiply(15, 8)
        │
        ▼
LANGCHAIN
Find multiply function
        │
        ▼
PYTHON
multiply(15, 8)
        │
        ▼
RESULT
120
        │
        ▼
LANGCHAIN
Send tool result + conversation
        │
        ▼
LLM
Decides: Call divide
        │
        ▼
TOOL CALL
divide(120, 3)
        │
        ▼
PYTHON
divide(120, 3)
        │
        ▼
RESULT
40
        │
        ▼
LANGCHAIN
Send result to LLM
        │
        ▼
LLM
Final Answer: 40
        │
        ▼
USER
```

This is what is happening behind:

```python
agent.invoke(...)
```

---

# 14. The Key Architecture

It is useful to think about the system as three major layers.

## Layer 1: Model Reasoning

The LLM decides:

```text
What does the user want?

What should I do next?

Which tool should I use?

Do I need another tool?

Am I finished?
```

---

## Layer 2: Framework Orchestration

LangChain manages:

```text
Tool schemas
Tool-call parsing
Tool registry
Function mapping
Tool execution
Message history
Tool results
Loop management
```

---

## Layer 3: Actual Execution

Python executes the real operation.

For example:

```python
multiply(15, 8)
```

or in production:

```python
query_database()
```

```python
call_api()
```

```python
send_email()
```

```python
create_ticket()
```

The architecture is:

```text
┌─────────────────────┐
│       LLM           │
│  Reasoning / Plan   │
└──────────┬──────────┘
           │
           ▼
┌─────────────────────┐
│      LangChain      │
│   Orchestration     │
└──────────┬──────────┘
           │
           ▼
┌─────────────────────┐
│   Python / Tools    │
│  Actual Execution   │
└─────────────────────┘
```

---

# 15. Why Agent Frameworks Are Important

Without a framework, developers would need to manually handle:

```text
Tool Registration
       ↓
Schema Generation
       ↓
API Requests
       ↓
Tool Call Parsing
       ↓
Function Lookup
       ↓
Function Execution
       ↓
Result Formatting
       ↓
Conversation History
       ↓
Loop Management
       ↓
Error Handling
```

Frameworks such as LangChain automate many of these repetitive tasks.

This allows developers to focus on:

* What the agent should do
* Which tools it should have
* How the tools should behave
* How the application should be secured
* How the agent should be monitored

The framework handles much of the underlying infrastructure.

---

# 16. The Most Important Mental Model

Remember this simple flow:

```text
User Question
      ↓
LLM Reasons
      ↓
Tool Call Request
      ↓
LangChain Parses
      ↓
Tool Registry Lookup
      ↓
Python Function Executes
      ↓
Tool Result
      ↓
LLM Receives Result
      ↓
LLM Reasons Again
      ↓
Another Tool?
    /       \
  Yes        No
   ↓          ↓
Repeat    Final Answer
```

This is the core of a tool-using AI agent.

The model does not directly execute your code.

The model **decides what should happen**.

The framework **orchestrates the process**.

The Python tools **perform the actual actions**.

This separation is one of the most important concepts to understand when building AI agents.

In the next part, we can continue from the second tool call and follow the process all the way to the final answer.
